google collab depedencies

In [1]:
!pip -q install bertopic
!pip -q install sastrawi
!pip -q install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 32.1 MB/s eta 0:00:00


In [2]:
!git clone -q -b gavriel-thesis https://github.com/ranslemus/topic_modeling_KBMI4.git
%cd topic_modeling_KBMI4

/content/topic_modeling_KBMI4


In [3]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.express as px
import random

from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from hdbscan.validity import validity_index

# for linux
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN

# for windows
# import umap as UMAP
# import hdbscan as HDBSCAN

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Device : cuda
GPU : Tesla T4


In [73]:
df = pd.read_csv("data/preprocessed_data_downsampled.csv")
df = df[df["year"]==2025]
df.head()

,reviewId,bank,score,year,text
3,629f06db-dc19-4a6b-a526-c5fa09933ed2,LIVIN_MANDIRI_REVIEWS,2,2025,kenapa di login tidak bisa ya malah muncul tul...
10,33535e95-15cb-49b3-bcb7-894957cb159d,WONDR_BNI_REVIEWS,1,2025,ngelag mulu deh
12,8091018d-801d-4a9f-a3ff-86491d781b1e,BRIMO_REVIEWS,1,2025,transaksi berhasil uang enggak masuk gimnaa si...
13,658c217f-74b2-4280-b07a-7ab529fd97a1,BCAMOBILE_REVIEWS,2,2025,sering keluar harus verifikasi lagi terus luma...
17,47ed6779-21fd-45bb-a5a6-c6d92ef93186,BCAMOBILE_REVIEWS,1,2025,malu ih bca mah


In [72]:
# df["word_count"] = df["text"].astype(str).str.split().apply(len)
# df = df[df["word_count"] >= 5].reset_index(drop=True)
# print(f"Total documents setelah filter: {len(df):,}")

Total documents setelah filter: 42,661


In [74]:
documents = df["text"].astype(str).tolist()

print(f"Total documents : {len(documents):,}")

Total documents : 49,342


# IndoBERT-base-p1

In [ ]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(MODEL_NAME)

model.to(device)

model.eval()

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: LazarusNLP/simcse-indobert-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(50000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    return torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    ) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

In [ ]:
def encode_documents(
    documents,
    batch_size=32,
    max_length=128
):

    embeddings = []

    with torch.no_grad():

        for i in tqdm(
            range(0, len(documents), batch_size)
        ):

            batch = documents[
                i:i+batch_size
            ]

            encoded_input = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            encoded_input = {
                k: v.to(device)
                for k, v in encoded_input.items()
            }

            model_output = model(**encoded_input)

            sentence_embeddings = mean_pooling(
                model_output,
                encoded_input["attention_mask"]
            )

            sentence_embeddings = (
                sentence_embeddings
                .cpu()
                .numpy()
            )

            embeddings.append(sentence_embeddings)

    return np.vstack(embeddings)

In [ ]:
embeddings = encode_documents(
    documents,
    batch_size=32,
    max_length=128
)

  0%|          | 0/1334 [00:00<?, ?it/s]

In [ ]:
print(embeddings.shape)

(42661, 768)


In [ ]:
embeddings[0]

array([ 1.31894362e+00,  9.60366786e-01,  1.30218878e-01, -6.16801903e-02,
        4.10509288e-01, -7.32329428e-01, -1.66571188e+00,  8.28113317e-01,
        8.47559512e-01,  4.62959141e-01, -1.03036702e+00, -1.15735888e+00,
       -9.49390411e-01, -1.97022617e-01, -4.13674146e-01, -5.59255257e-02,
       -8.73682320e-01, -3.98348600e-01,  9.12975729e-01,  1.10419989e-01,
        1.62639177e+00,  6.93946123e-01,  2.03165129e-01, -3.73411298e-01,
       -7.23186851e-01, -1.41948688e+00,  2.93903202e-01, -4.36818421e-01,
       -4.34622020e-01, -1.46654904e-01,  3.04300129e-01,  3.05604726e-01,
        1.74515486e+00,  6.05405420e-02,  2.28680789e-01, -9.99139808e-03,
        6.52468562e-01,  1.10717797e+00, -1.17894602e+00,  1.68272883e-01,
        4.44559038e-01, -1.43568218e-01,  7.49856293e-01, -8.61199439e-01,
       -4.08587456e-01,  2.14764491e-01,  1.07191110e+00,  1.40870261e+00,
        1.30559671e+00,  1.03162718e+00, -1.46915901e+00, -1.55424818e-01,
       -5.29873371e-01,  

In [ ]:
norms = np.linalg.norm(embeddings, axis=1)

print("Minimum Norm :", norms.min())
print("Maximum Norm :", norms.max())
print("Average Norm :", norms.mean())
print("Std Norm :", norms.std())

Minimum Norm : 15.018309
Maximum Norm : 26.08848
Average Norm : 21.370525
Std Norm : 1.9197338


In [ ]:
print("NaN :", np.isnan(embeddings).sum())
print("Inf :", np.isinf(embeddings).sum())

NaN : 0
Inf : 0


In [ ]:
# np.save(
#     "embeddings/indobert_embeddings_downsampled_cutted.npy",
#     embeddings
# )

# simcse-indobert-base

In [75]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "LazarusNLP/simcse-indobert-base"
embedding_model = SentenceTransformer(MODEL_NAME, device=device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [76]:
embeddings = embedding_model.encode(
    documents,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding Shape:", embeddings.shape)

Batches:   0%|          | 0/771 [00:00<?, ?it/s]

Embedding Shape: (49342, 768)


# BERTopic

In [102]:
# embeddings = np.load("embeddings/indobert_embeddings_downsampled_full.npy")

print("Embedding Shape :", embeddings.shape)

Embedding Shape : (49342, 768)


stopwords

In [103]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords as nltk_stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [104]:
sastrawi_stopwords = StopWordRemoverFactory().get_stop_words()

# extra_particles = ["banget", "terus", "padahal", "sih", "aja", "saja", "dong", "deh", "ya", "kok", "biar", "gitu", "nih", "loh", "mau", "sudah", "belum"]
# sastrawi_stopwords_extended = sastrawi_stopwords

vectorizer_model = CountVectorizer(
  ngram_range=(1,2),
  stop_words=sastrawi_stopwords,
  token_pattern=r"(?u)\b[^\d\W]+\b",
  min_df=2,
  )

baseline UMAP for testing purpose

In [105]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    metric="cosine",
    min_dist=0.0,
    random_state=42
)

baseline HDBSCAN

In [106]:
hdbscan_model = HDBSCAN(
    min_cluster_size=30,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="leaf",
    prediction_data=True
)

In [107]:
from bertopic.vectorizers import ClassTfidfTransformer

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

In [108]:
topic_model = BERTopic(
    embedding_model=None,
    calculate_probabilities=False,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
)

In [109]:
topics, probabilities = topic_model.fit_transform(
    documents,
    embeddings
)

2026-08-11 08:34:44,501 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-11 08:34:46,515 - BERTopic - Dimensionality - Completed ✓
2026-08-11 08:34:46,520 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-11 08:34:47,033 - BERTopic - Cluster - Completed ✓
2026-08-11 08:34:47,053 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-11 08:34:50,623 - BERTopic - Representation - Completed ✓


# Evaluation

Basic Statistics

In [110]:
topic_info = topic_model.get_topic_info()

topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,30817,-1_bayar_banking_mau transaksi_jaringan,"[bayar, banking, mau transaksi, jaringan, bri,...",[nih bagaimana sih wonder bni dari tadi mau ko...
1,0,523,0_uninstall_cache_uninstal_instal,"[uninstall, cache, uninstal, instal, clear, in...",[sepertinya memang ada masalah di aplikasi ter...
2,1,515,1_otp_kode otp_otp enggak_otp nya,"[otp, kode otp, otp enggak, otp nya, kode, otp...",[kirim kode otp berkali kali tapi enggak mener...
3,2,503,2_pasword_sandi_username_password,"[pasword, sandi, username, password, pasword b...",[tiba tiba tidak bisa log ini pasword salah mu...
4,3,333,3_nama_scroll_search_pencarian,"[nama, scroll, search, pencarian, manual, keti...",[tolong saya baru saja kena update otomatis fi...
5,4,324,4_android_oppo_os_redmi,"[android, oppo, os, redmi, vivo, operasi, sams...",[per hari ini tanggal 24 desember 2025 hp saya...
6,5,312,5_ribu_sisa_biaya admin_potongan,"[ribu, sisa, biaya admin, potongan, ribu padah...",[suka ada potongan enggak jelas ke tl padahal ...
7,6,297,6_kartu debit_kartu atm_debit_debit kredit,"[kartu debit, kartu atm, debit, debit kredit, ...",[bikin rekening baru harus pakai aplikasi baru...
8,7,263,7_tunai_tarik tunai_setor tunai_setor,"[tunai, tarik tunai, setor tunai, setor, tunai...",[enggak bisa tarik tunai atau setor tunai tanp...
9,8,238,8_akun brimo_username password_terdaftar brimo...,"[akun brimo, username password, terdaftar brim...",[saya tidak memiliki akun brimo sebelumnya tap...


In [111]:
num_topics = len(topic_info) - 1

outlier_count = (np.array(topics) == -1).sum()

outlier_percentage = (
    outlier_count / len(topics)
) * 100

print(f"Topics              : {num_topics}")
print(f"Outliers            : {outlier_count:,}")
print(f"Outlier Percentage  : {outlier_percentage:.2f}%")

Topics              : 228
Outliers            : 30,817
Outlier Percentage  : 62.46%


Topic Size

In [112]:
topic_info[["Topic","Count"]]

,Topic,Count
0,-1,30817
1,0,523
2,1,515
3,2,503
4,3,333
...,...,...
224,223,31
225,224,31
226,225,31
227,226,31


Top Words

In [113]:
topic_summary = []
for topic in topic_info.Topic:
    if topic == -1:
        continue
    words = ", ".join([w for w, _ in topic_model.get_topic(topic)[:10]])
    topic_summary.append({"Topic": topic, "Count": topic_info.loc[topic_info.Topic==topic, "Count"].values[0], "Top Words": words})

topic_summary_df = pd.DataFrame(topic_summary).sort_values("Count", ascending=False)
topic_summary_df

,Topic,Count,Top Words
0,0,523,"uninstall, cache, uninstal, instal, clear, ins..."
1,1,515,"otp, kode otp, otp enggak, otp nya, kode, otp ..."
2,2,503,"pasword, sandi, username, password, pasword be..."
3,3,333,"nama, scroll, search, pencarian, manual, ketik..."
4,4,324,"android, oppo, os, redmi, vivo, operasi, samsu..."
...,...,...,...
223,223,31,"pulsa ribu, jelas pulsa, habis pulsa, pulsa cu..."
224,224,31,"mencoba besok, besok besok, besok, suruh menco..."
225,225,31,"keren, keren keren, luar biasa, biasa biasa, b..."
226,226,31,"persulit, daftar persulit, persulit apk, persu..."


silhoutte score

In [115]:
from sklearn.metrics import silhouette_score

mask = np.array(topics) != -1

silhouette = silhouette_score(
    topic_model.umap_model.embedding_[mask],
    np.array(topics)[mask]
)

print(f"Silhouette Score : {silhouette:.4f}")

Silhouette Score : 0.5844


In [116]:
from itertools import chain

top_n = 10
topic_words = []

for topic in topic_info.Topic:
    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
    ]
    topic_words.append(words)

unique_words = len(
    set(chain.from_iterable(topic_words))
)

total_words = len(topic_words) * top_n
topic_diversity = unique_words / total_words

print(f"Topic Diversity : {topic_diversity:.4f}")

Topic Diversity : 0.8961


Representative Reviews

In [117]:
topic_info = topic_model.get_topic_info()
top_10_topics = topic_info[topic_info.Topic != -1].nlargest(10, "Count")["Topic"].tolist()

print("=== TOP 10 TOPIK PALING REPRESENTATIF ===")

for topic_id in top_10_topics:
    # Ambil ukuran klaster asli
    cluster_size = topic_info.loc[topic_info.Topic == topic_id, "Count"].values[0]

    # Ambil kata kunci utama topik untuk mempermudah pembacaan aspek
    keywords = ", ".join([w for w, _ in topic_model.get_topic(topic_id)[:5]])

    # Ambil dokumen yang secara matematis paling dekat dengan centroid klaster (Bawaan BERTopic)
    rep_docs = topic_model.get_representative_docs(topic_id)

    print("\n" + "=" * 120)
    print(f"TOPIC {topic_id} | CLUSTER SIZE: {cluster_size}")
    print(f"KEYWORDS : {keywords}")
    print("=" * 120)

    # BERTopic menyimpan maksimum 3 representative docs per topik secara default
    for i, doc in enumerate(rep_docs, 1):
        print(f"{i}. {doc}")

=== TOP 10 TOPIK PALING REPRESENTATIF ===

TOPIC 0 | CLUSTER SIZE: 523
KEYWORDS : uninstall, cache, uninstal, instal, clear
1. sepertinya memang ada masalah di aplikasi tersebut 2x delete instal aplikasi di awal memang berjalan normal tapi beberapa jam kemudian kembali hanya stuck di logo saja tanpa bisa login dan akan kembali normal setelah hp di matikan terlebih dahulu di mode pesawat sudah hapus cache tanpa wifi tanpa sim 2 only sim 1 yang terdaftar dll masalah terus terulang stuck di logo awal tanpa bisa login tolong di perbaiki ini sangat mengganggu saat akan melakukan transaksi
2. sesudah update versi terbaru tiap mau transaksi selalu ada pesan pop up silakan tutup aplikasi lainnya agar bisa kembali bertransaksi sudah di tutup juga sudah restart hp sudah coba uninstall dan install ulang sudah ke kc bri dipatiukur mereka nyerah dan menyarankan hp nya di bawa ke konter terus coba lagi ke kc jl ir h djuanda yang mana kantor lebih besar sampai itu nya turun tangan dan tetap masalah e

NPMI

In [118]:
analyzer = topic_model.vectorizer_model.build_analyzer()

In [119]:
doc.split()

['sering',
 'keluar',
 'sendiri',
 'peliharaan',
 'mulu',
 'kacau',
 'padahal',
 'buka',
 'buka',
 'doang',
 'malah',
 'keluar',
 'sendiri']

In [120]:
tokenized_docs = [
    analyzer(doc)
    for doc in documents
]

In [121]:
from gensim.corpora import Dictionary

dictionary = Dictionary(tokenized_docs)
topic_words = []

for topic in topic_info.Topic:

    if topic == -1:
        continue

    words = []

    for word, score in topic_model.get_topic(topic):
        if word in dictionary.token2id:
            words.append(word)
    # Need at least 2 words for coherence
    if len(words) >= 2:
        topic_words.append(words)

In [122]:
# sanity check
print(f"Valid Topics : {len(topic_words)}")

print()

print(topic_words[:3])

Valid Topics : 228

[['uninstall', 'cache', 'uninstal', 'instal', 'clear', 'instal ulang', 'hapus', 'install', 'ulang tetap', 'install ulang'], ['otp', 'kode otp', 'otp enggak', 'otp nya', 'kode', 'otp masuk', 'kirim otp', 'meminta kode', 'menerima kode', 'meminta otp'], ['pasword', 'sandi', 'username', 'password', 'pasword benar', 'username password', 'password benar', 'username sama', 'salah', 'padahal benar']]


In [123]:
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence="c_npmi"
)

npmi = coherence_model.get_coherence()
print(f"NPMI : {npmi:.4f}")

NPMI : -0.0826


DBCV

In [124]:
mask = np.array(topics) != -1
X = topic_model.umap_model.embedding_[mask].astype(np.float64)
labels = np.array(topics)[mask]

dbcv_score = validity_index(X, labels)
print(f"DBCV : {dbcv_score:.4f}")

DBCV : 0.3377


In [125]:
import pandas as pd
from scipy.stats import chi2_contingency

df["topic"] = topics

# 1. Baseline: proporsi tiap bank di keseluruhan korpus
baseline = df["bank"].value_counts(normalize=True) * 100
print("Proporsi bank di keseluruhan korpus (baseline):")
print(baseline.round(2))
print()

# 2. Proporsi tiap bank DI DALAM tiap topik
crosstab = pd.crosstab(df["topic"], df["bank"], normalize="index") * 100
crosstab = crosstab.round(2)

# 3. Hitung "lift" = proporsi di topik / proporsi baseline
#    >1 artinya over-represented di topik itu, <1 artinya under-represented
lift = crosstab.copy()
for bank in baseline.index:
    lift[bank] = crosstab[bank] / baseline[bank]

# 4. Tandai topik yang "njomplang" (deviasi lift > 1.5x atau < 0.5x dari baseline)
def flag_imbalance(row):
    return any(row > 1.5) or any(row < 0.5)

lift["is_imbalanced"] = lift[baseline.index].apply(flag_imbalance, axis=1)

# gabung count per topik biar gampang liat mana yang topik "besar" (bukan cuma noise kecil)
topic_sizes = df[df["topic"] != -1]["topic"].value_counts()
lift["topic_size"] = lift.index.map(topic_sizes)

result = lift[lift.index != -1].sort_values("is_imbalanced", ascending=False)
print(result[list(baseline.index) + ["is_imbalanced", "topic_size"]])

Proporsi bank di keseluruhan korpus (baseline):
bank
BRIMO_REVIEWS            28.87
WONDR_BNI_REVIEWS        27.62
LIVIN_MANDIRI_REVIEWS    25.69
BCAMOBILE_REVIEWS        17.82
Name: proportion, dtype: float64

bank   BRIMO_REVIEWS  WONDR_BNI_REVIEWS  LIVIN_MANDIRI_REVIEWS  \
topic                                                            
114         2.248045           0.127103               0.409886   
132         0.196024           1.161670               2.276754   
123         0.314816           1.975346               1.132344   
124         3.463326           0.000000               0.000000   
206         1.364204           0.329164               1.769165   
...              ...                ...                    ...   
127         1.475030           0.804623               0.720901   
125         1.154326           1.139943               0.936937   
113         0.911547           1.143564               1.024520   
112         1.373555           0.624289               1.006614 

finding the best settings for both HDBSCAN and UMAP

In [126]:
# import itertools

# param_grid = {
#     "min_cluster_size": [50, 75, 100],
#     "min_samples": [5, 10],
#     "cluster_selection_epsilon": [0.0, 0.05, 0.1],
# }

# results = []
# combos = list(itertools.product(*param_grid.values()))
# print(f"Total kombinasi yang dicoba: {len(combos)}")

# for mcs, ms, eps in combos:
#     hdbscan_test = HDBSCAN(
#         min_cluster_size=mcs,
#         min_samples=ms,
#         metric="euclidean",
#         cluster_selection_method="leaf",
#         cluster_selection_epsilon=eps,
#         prediction_data=True,
#     )
#     tm = BERTopic(
#         embedding_model=None, calculate_probabilities=False,
#         vectorizer_model=vectorizer_model, ctfidf_model=ctfidf_model,
#         umap_model=umap_model, hdbscan_model=hdbscan_test, verbose=False,
#     )
#     tpcs, _ = tm.fit_transform(documents, embeddings)

#     ti = tm.get_topic_info()
#     n_topics = len(ti) - 1
#     outlier_pct = (np.array(tpcs) == -1).sum() / len(tpcs) * 100
#     max_share = ti[ti.Topic != -1]["Count"].max() / len(tpcs) * 100 if n_topics > 0 else 0
#     mask = np.array(tpcs) != -1
#     sil = silhouette_score(tm.umap_model.embedding_[mask], np.array(tpcs)[mask]) if len(set(np.array(tpcs)[mask])) > 1 else float("nan")

#     row = {"min_cluster_size": mcs, "min_samples": ms, "epsilon": eps,
#            "topics": n_topics, "outlier_%": round(outlier_pct, 2),
#            "max_topic_share_%": round(max_share, 2), "silhouette": round(sil, 4)}
#     results.append(row)
#     print(row)

# results_df = pd.DataFrame(results).sort_values("outlier_%")
# results_df